In [1]:
# ================================================================
# DSC GAMMAFEST 2026 — Football Score Prediction  v5
#
# ANALISIS skor 2.906 (v4):
#   Bayesian Opt pakai Poisson murni yang TIDAK COCOK dengan data:
#   • 0-0: aktual 7.4% vs Poisson 4.4%  → under-predict 3pp!
#   • 1-1: aktual 9.4% vs Poisson 10.7% → over-predict 1.3pp
#   • 2-2: aktual 3.9% vs Poisson 6.5%  → over-predict 2.6pp!
#   → Bayesian opt pakai distribusi yang salah → pilihan scoreline suboptimal
#
# PERBAIKAN v5:
#   1. Dixon-Coles correction: fit ρ dari data → perbaiki P(0-0), P(1-1) dll
#   2. Isotonic calibration: koreksi systematic bias λ dari model
#   3. CatBoost: model ke-3 untuk ensemble yang lebih diverse
#   4. Time-based sample weights lebih agresif (test=2011-2026)
#   5. GD-direction features lebih kaya
# ================================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import poisson as sp_poisson
from scipy.optimize import minimize_scalar
from sklearn.model_selection import KFold
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
    print("CatBoost found ✓")
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost not found — install with: pip install catboost")
    print("Continuing with LGB+XGB only...")

SEED = 42
np.random.seed(SEED)

# ================================================================
# CONFIG
# ================================================================
DATA_DIR   = Path('.')
N_FOLDS    = 5
MAX_TREES  = 3000
MAX_GOALS  = 10

# ================================================================
# 1. LOAD DATA
# ================================================================
train  = pd.read_csv(DATA_DIR / 'train.csv',  parse_dates=['date'])
test   = pd.read_csv(DATA_DIR / 'test.csv',   parse_dates=['date'])
sample = pd.read_csv(DATA_DIR / 'sample submission.csv')

print(f"Train : {train.shape[0]:,} rows | {train['date'].min().date()} → {train['date'].max().date()}")
print(f"Test  : {test.shape[0]:,} rows  | {test['date'].min().date()} → {test['date'].max().date()}")

# ================================================================
# 2. AW-MAE
# ================================================================
TOURNAMENT_WEIGHTS = {
    'FIFA World Cup'         : 2.00,
    'AFC Asian Cup'          : 1.80,
    'UEFA Euro'              : 1.80,
    'Copa América'           : 1.80,
    'African Cup of Nations' : 1.80,
    'CONCACAF Gold Cup'      : 1.80,
    'OFC Nations Cup'        : 1.80,
    'Friendly'               : 0.96,
}
DEFAULT_WEIGHT = 1.20

def awmae(df, pt, po):
    tg  = df['team_goals'].values.astype(float)
    og  = df['opp_goals'].values.astype(float)
    tp  = np.clip(np.asarray(pt, float), 0, None)
    op  = np.clip(np.asarray(po, float), 0, None)
    tr  = np.round(tp); orr = np.round(op)
    mae = (np.abs(tg - tp) + np.abs(og - op)) / 2
    ex  = ((tr == tg) & (orr == og)).astype(float)
    oc  = (np.sign(tg - og) == np.sign(tr - orr)).astype(float)
    gd  = ((tg - og) == (tr - orr)).astype(float)
    pen = 0.30*(1-ex) + 0.25*(1-oc) + 0.15*(1-gd)
    mul = np.where(oc == 1, 1.0, 1.5)
    lss = ((mae + pen) * mul) ** 1.5
    w   = df['tournament'].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_WEIGHT).values
    return float(np.sum(lss * w) / np.sum(w))

def awmae_int(df, pred_t, pred_o):
    tg  = df['team_goals'].values.astype(float)
    og  = df['opp_goals'].values.astype(float)
    pt  = pred_t.astype(float); po = pred_o.astype(float)
    mae = (np.abs(tg - pt) + np.abs(og - po)) / 2
    ex  = ((pt == tg) & (po == og)).astype(float)
    oc  = (np.sign(tg - og) == np.sign(pt - po)).astype(float)
    gd  = ((tg - og) == (pt - po)).astype(float)
    pen = 0.30*(1-ex) + 0.25*(1-oc) + 0.15*(1-gd)
    mul = np.where(oc == 1, 1.0, 1.5)
    lss = ((mae + pen) * mul) ** 1.5
    w   = df['tournament'].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_WEIGHT).values
    return float(np.sum(lss * w) / np.sum(w))

# ================================================================
# 3. PRECOMPUTE AW-MAE LOSS TENSOR
# ================================================================
print("Precomputing AW-MAE loss tensor...")
G     = MAX_GOALS + 1
goals = np.arange(G)
ti = goals[:,None,None,None].astype(float)
tj = goals[None,:,None,None].astype(float)
pi = goals[None,None,:,None].astype(float)
pj = goals[None,None,None,:].astype(float)
mae_t   = (np.abs(ti-pi) + np.abs(tj-pj)) / 2
exact_t = ((pi==ti) & (pj==tj)).astype(float)
oc_t    = (np.sign(ti-tj) == np.sign(pi-pj)).astype(float)
gd_t    = ((ti-tj) == (pi-pj)).astype(float)
pen_t   = 0.30*(1-exact_t) + 0.25*(1-oc_t) + 0.15*(1-gd_t)
mul_t   = np.where(oc_t == 1, 1.0, 1.5)
BASE_LOSS = ((mae_t + pen_t) * mul_t) ** 1.5
BL_FLAT   = BASE_LOSS.reshape(G*G, G*G)
print(f"  Shape: {BASE_LOSS.shape} ✓")

# ================================================================
# 4. DIXON-COLES CORRECTION — KEY FIX
#
# Standard Poisson P(i,j) = Poisson(i,λt) × Poisson(j,λo)
# Dixon-Coles adjusts P(i,j) for low-scoring matches:
#   τ(0,0) = 1 - λt*λo*ρ   (if ρ<0: more 0-0 draws)
#   τ(1,0) = 1 + λo*ρ
#   τ(0,1) = 1 + λt*ρ
#   τ(1,1) = 1 - ρ
#   τ(i,j) = 1  otherwise
#
# We fit ρ on training data using MLE
# ================================================================
def dc_correction(lam_t, lam_o, rho):
    """
    Dixon-Coles correction factor τ for all (i,j) pairs.
    Returns array (N, G, G).
    """
    N = len(lam_t)
    tau = np.ones((N, G, G), dtype=float)
    # (0,0)
    tau[:, 0, 0] = 1 - lam_t * lam_o * rho
    # (1,0)
    tau[:, 1, 0] = 1 + lam_o * rho
    # (0,1)
    tau[:, 0, 1] = 1 + lam_t * rho
    # (1,1)
    tau[:, 1, 1] = 1 - rho
    return np.maximum(tau, 1e-10)   # avoid negative probabilities


def dc_joint_pmf(lam_t, lam_o, rho):
    """
    Dixon-Coles joint PMF: P(team=i, opp=j | λt, λo, ρ)
    Returns (N, G, G), normalized.
    """
    # Independent Poisson
    p_t = sp_poisson.pmf(goals[None, :], lam_t[:, None])   # (N, G)
    p_o = sp_poisson.pmf(goals[None, :], lam_o[:, None])   # (N, G)
    p   = p_t[:, :, None] * p_o[:, None, :]                # (N, G, G)

    # Apply Dixon-Coles correction
    tau = dc_correction(lam_t, lam_o, rho)
    p   = p * tau

    # Renormalize (correction changes total mass slightly)
    p   = np.maximum(p, 1e-12)
    p  /= p.sum(axis=(1, 2), keepdims=True)
    return p


def dc_neg_loglik(rho, lam_t_oof, lam_o_oof, true_t, true_o):
    """Negative log-likelihood of Dixon-Coles model for fitting ρ."""
    N = len(lam_t_oof)
    # Compute log P(true_t, true_o) for each instance
    p_t = sp_poisson.pmf(true_t, lam_t_oof)
    p_o = sp_poisson.pmf(true_o, lam_o_oof)
    joint_p = p_t * p_o

    # Correction for low-scoring matches
    mask_00 = (true_t == 0) & (true_o == 0)
    mask_10 = (true_t == 1) & (true_o == 0)
    mask_01 = (true_t == 0) & (true_o == 1)
    mask_11 = (true_t == 1) & (true_o == 1)

    correction = np.ones(N)
    correction[mask_00] = 1 - lam_t_oof[mask_00] * lam_o_oof[mask_00] * rho
    correction[mask_10] = 1 + lam_o_oof[mask_10] * rho
    correction[mask_01] = 1 + lam_t_oof[mask_01] * rho
    correction[mask_11] = 1 - rho

    joint_p = np.maximum(joint_p * correction, 1e-15)
    return -np.sum(np.log(joint_p))


# ================================================================
# 5. FEATURE ENGINEERING
# ================================================================
def add_base(df):
    df = df.copy()
    df['year']        = df['date'].dt.year
    df['month']       = df['date'].dt.month
    df['is_modern']   = (df['year'] >= 2000).astype(int)
    df['is_recent']   = (df['year'] >= 2008).astype(int)
    df['is_women']    = (df['gender'] == 'W').astype(int)
    df['tw']          = df['tournament'].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_WEIGHT)
    df['is_hs']       = (df['tw'] >= 1.8).astype(int)
    df['is_wc']       = (df['tournament'] == 'FIFA World Cup').astype(int)
    df['is_friendly'] = (df['tournament'] == 'Friendly').astype(int)
    df['same_conf']   = (df['confederation_team'] == df['confederation_opp']).astype(int)
    df['cm']          = df['confederation_team'] + '_vs_' + df['confederation_opp']
    df['alt']         = df['altitude_venue'].clip(lower=0).fillna(100)
    df['temp']        = df['temperature_venue'].fillna(20)
    df['is_high_alt'] = (df['alt'] > 2000).astype(int)
    df['dist_adv']    = df['distance_travel_opp'].fillna(0) - df['distance_travel_team'].fillna(0)
    df['has_dist']    = df['distance_travel_team'].notna().astype(int)
    df['women_home']  = df['is_women'] * df['is_home']
    df['alt_home']    = df['is_high_alt'] * df['is_home']
    df['hs_home']     = df['is_hs'] * df['is_home']
    for c in ['gdp_per_capita_team', 'gdp_per_capita_opp',
              'population_team',     'population_opp']:
        df[f'l{c}'] = np.log1p(df[c].fillna(df[c].median()))
    df['gdp_d'] = df['lgdp_per_capita_team'] - df['lgdp_per_capita_opp']
    df['pop_r'] = df['lpopulation_team']      - df['lpopulation_opp']
    return df

train = add_base(train)
test  = add_base(test)

# ================================================================
# 6. TARGET ENCODING
# ================================================================
def _fallback(tr, ts, gc, tc):
    gm = tr[tc].mean()
    cc = ('confederation_team' if gc == 'team'
          else 'confederation_opp' if gc == 'opponent' else None)
    if cc and cc in ts.columns:
        return ts[cc].map(tr.groupby(cc)[tc].mean()).fillna(gm)
    return gm

def smooth_te(tr, ts, gc, tc, nc, sm=20, nf=5):
    gm  = tr[tc].mean()
    grp = tr.groupby(gc)[tc].agg(['sum', 'count'])
    grp['e'] = (grp['sum'] + gm * sm) / (grp['count'] + sm)
    ts[nc] = ts[gc].map(grp['e']).fillna(_fallback(tr, ts, gc, tc)).values
    enc = np.full(len(tr), gm, dtype=float)
    for ti, vi in KFold(nf, shuffle=True, random_state=SEED).split(tr):
        fg = tr.iloc[ti].groupby(gc)[tc].agg(['sum', 'count'])
        fg['e'] = (fg['sum'] + gm * sm) / (fg['count'] + sm)
        enc[vi] = tr.iloc[vi][gc].map(fg['e']).fillna(gm).values
    tr[nc] = enc
    return tr, ts

def recency_te(tr, ts, gc, tc, nc, hl=365*3, sm=15, nf=3):
    ref = tr['date'].max()
    tr_ = tr.copy()
    tr_['dw'] = np.exp(-(ref - tr_['date']).dt.days / hl)
    gm  = tr[tc].mean()
    def wm(x, dw): sw = dw.sum(); return (x*dw).sum()/sw if sw > 0 else gm
    grp = tr_.groupby(gc).apply(
        lambda x: (wm(x[tc].values, x['dw'].values)*len(x) + gm*sm) / (len(x)+sm))
    ts[nc] = ts[gc].map(grp).fillna(_fallback(tr, ts, gc, tc)).values
    enc = np.full(len(tr), gm, dtype=float)
    for ti, vi in KFold(nf, shuffle=True, random_state=SEED).split(tr):
        fold = tr_.iloc[ti]
        fg   = fold.groupby(gc).apply(
            lambda x: (wm(x[tc].values, x['dw'].values)*len(x) + gm*sm) / (len(x)+sm))
        enc[vi] = tr.iloc[vi][gc].map(fg).fillna(gm).values
    tr[nc] = enc
    return tr, ts

print("Building encodings...")
# Score and GD encodings
for tc, s in [('team_goals', 'ts'), ('opp_goals', 'tc')]:
    train, test = smooth_te(train, test, 'team',     tc, f'te_{s}', 15)
    train, test = smooth_te(train, test, 'opponent', tc, f'oe_{s}', 15)
for tc, s in [('team_goals', 't'), ('opp_goals', 'o')]:
    train, test = smooth_te(train, test, 'tournament', tc, f'tu{s}', 30)
    train, test = smooth_te(train, test, 'cm',         tc, f'cm{s}', 50)
train, test = smooth_te(train, test, 'venue_country', 'team_goals', 'vct', 20)
train['gdt'] = train['team_goals'] - train['opp_goals']
train, test  = smooth_te(train, test, 'team',     'gdt', 'te_gd', 15)
train, test  = smooth_te(train, test, 'opponent', 'gdt', 'oe_gd', 15)

print("Building recency encodings...")
for gc, tc, nc in [('team','team_goals','te_rw'), ('opponent','team_goals','oe_rw'),
                   ('team','gdt','te_gd_rw'),     ('opponent','gdt','oe_gd_rw')]:
    train, test = recency_te(train, test, gc, tc, nc)

# Zero-rate encodings
train['tz'] = (train['team_goals'] == 0).astype(float)
train['oz'] = (train['opp_goals']  == 0).astype(float)
train, test = smooth_te(train, test, 'team',     'tz', 'te_z', 15)
train, test = smooth_te(train, test, 'opponent', 'oz', 'oe_z', 15)
train, test = recency_te(train, test, 'team',     'tz', 'te_z_rw')
train, test = recency_te(train, test, 'opponent', 'oz', 'oe_z_rw')

# Tournament × team goal encoding (captures "high-stakes team strength")
train['gdt_hs'] = train['gdt'] * train['is_hs']
train['gdt_wc'] = train['gdt'] * train['is_wc']
train, test = smooth_te(train, test, 'team', 'gdt_hs', 'te_gd_hs', sm=30)
train, test = smooth_te(train, test, 'team', 'gdt_wc', 'te_gd_wc', sm=50)

# Aggregate stats
gm  = train['team_goals'].mean()
ha  = train[train['is_home'] == 1].groupby('team')['team_goals'].mean()
aa  = train[train['is_home'] == 0].groupby('team')['team_goals'].mean()
twr = (train['team_goals'] > train['opp_goals']).groupby(train['team']).mean()
owr = (train['team_goals'] > train['opp_goals']).groupby(train['opponent']).mean()
tdr = (train['team_goals'] == train['opp_goals']).groupby(train['team']).mean()
# High-stakes win rate
train_hs = train[train['is_hs'] == 1]
twr_hs   = (train_hs['team_goals'] > train_hs['opp_goals']).groupby(train_hs['team']).mean()

for df in [train, test]:
    df['ha']          = df['team'].map(ha).fillna(gm)
    df['aa']          = df['team'].map(aa).fillna(gm)
    df['twr']         = df['team'].map(twr).fillna(0.39)
    df['owr']         = df['opponent'].map(owr).fillna(0.39)
    df['tdr']         = df['team'].map(tdr).fillna(0.22)
    df['twr_hs']      = df['team'].map(twr_hs).fillna(0.39)
    df['eff_rate']    = df['is_home'] * df['ha'] + (1 - df['is_home']) * df['aa']
    # Interaction features
    df['avd']         = df['te_ts']    - df['oe_tc']
    df['dva']         = df['oe_ts']    - df['te_tc']
    df['sd']          = df['te_ts']    - df['oe_ts']
    df['xgd']         = df['avd']      - df['dva']
    df['wrd']         = df['twr']      - df['owr']
    df['gdd']         = df['te_gd']    - df['oe_gd']
    df['rw_diff']     = df['te_rw']    - df['oe_rw']
    df['rw_gdd']      = df['te_gd_rw'] - df['oe_gd_rw']
    df['z_adv']       = df['oe_z']     - df['te_z']
    df['z_rw_adv']    = df['oe_z_rw']  - df['te_z_rw']
    # High-stakes direction
    df['gd_hs_diff']  = df['te_gd_hs'] - df['oe_gd']
    df['gd_wc_diff']  = df['te_gd_wc'] - df['oe_gd']

le = LabelEncoder()
le.fit(pd.concat([train['confederation_team'], train['confederation_opp'],
                  test['confederation_team'],  test['confederation_opp']]).unique())
for df in [train, test]:
    df['ctl'] = le.transform(df['confederation_team'])
    df['col'] = le.transform(df['confederation_opp'])

# ================================================================
# 7. FEATURE LIST
# ================================================================
FEATURES = [
    'te_ts','te_tc','oe_ts','oe_tc',
    'te_gd','oe_gd','gdd',
    'te_rw','oe_rw','rw_diff',
    'te_gd_rw','oe_gd_rw','rw_gdd',
    'te_z','oe_z','te_z_rw','oe_z_rw','z_adv','z_rw_adv',
    'te_gd_hs','te_gd_wc','gd_hs_diff','gd_wc_diff',
    'ha','aa','eff_rate','twr','owr','tdr','twr_hs',
    'avd','dva','sd','xgd','wrd',
    'tut','tuo','tw','is_hs','is_wc','is_friendly',
    'cmt','cmo','ctl','col','same_conf',
    'vct',
    'is_home','neutral','is_women','women_home','alt_home','hs_home',
    'alt','temp','is_high_alt','dist_adv','has_dist',
    'lgdp_per_capita_team','lgdp_per_capita_opp','gdp_d',
    'lpopulation_team','lpopulation_opp','pop_r',
    'year','month','is_modern','is_recent',
]

X_tr = train[FEATURES].fillna(0)
X_te = test[FEATURES].fillna(0)
print(f"\nFeatures: {len(FEATURES)} | NaN train: {X_tr.isnull().sum().sum()} | NaN test: {X_te.isnull().sum().sum()}")

# ================================================================
# 8. SAMPLE WEIGHTS — lebih agresif untuk data modern
# ================================================================
sw = np.ones(len(train), dtype=float)
sw[train['year'].values >= 2000] *= 2.0   # v5: lebih agresif
sw[train['year'].values >= 2005] *= 1.5
sw[train['year'].values >= 2008] *= 1.2   # paling dekat dengan test era
sw[train['is_hs'].values == 1]   *= 1.5
sw[train['is_wc'].values == 1]   *= 1.5

# ================================================================
# 9. MODEL PARAMS
# ================================================================
LGB_BASE = dict(
    n_estimators=MAX_TREES,
    learning_rate=0.02,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)

LGB_POIS = {
    **LGB_BASE,
    'objective': 'poisson',
}

LGB_GD = {
    **LGB_BASE,
    'objective': 'regression_l1',
}

LGB_BIN = {
    **LGB_BASE,
    'objective': 'binary',
    'n_estimators': min(MAX_TREES, 1500),
}

XGB_BASE = dict(
    n_estimators=MAX_TREES,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=150,
    random_state=SEED,
    verbosity=0,
    n_jobs=-1,
)

XGB_POIS = {
    **XGB_BASE,
    'objective': 'count:poisson',
    'eval_metric': 'poisson-nloglik',
}

XGB_GD = {
    **XGB_BASE,
    'objective': 'reg:absoluteerror',
    'eval_metric': 'mae',
}
# ================================================================
# 10. CV HELPERS
# ================================================================
def cv_lgb_reg(X, y, Xt, params, sw, label, nf=N_FOLDS, clip=True):
    kf = KFold(nf, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X)); pred = np.zeros(len(Xt))
    for f, (ti, vi) in enumerate(kf.split(X)):
        m = lgb.LGBMRegressor(**params)
        m.fit(X.iloc[ti], y.iloc[ti], sample_weight=sw[ti],
              eval_set=[(X.iloc[vi], y.iloc[vi])],
              callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)])
        oof[vi] = m.predict(X.iloc[vi]); pred += m.predict(Xt) / nf
        print(f"  [{label}] f{f+1} MAE={np.abs(oof[vi]-y.iloc[vi]).mean():.4f} i={m.best_iteration_}")
    if clip: oof = np.maximum(oof, 0); pred = np.maximum(pred, 0)
    print(f"  [{label}] OOF MAE={np.abs(oof-y).mean():.4f}\n")
    return oof, pred

def cv_lgb_bin(X, y, Xt, params, sw, label, nf=N_FOLDS):
    kf = KFold(nf, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X)); pred = np.zeros(len(Xt))
    for f, (ti, vi) in enumerate(kf.split(X)):
        m = lgb.LGBMClassifier(**params)
        m.fit(X.iloc[ti], y.iloc[ti], sample_weight=sw[ti],
              eval_set=[(X.iloc[vi], y.iloc[vi])],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
        oof[vi] = m.predict_proba(X.iloc[vi])[:, 1]
        pred   += m.predict_proba(Xt)[:, 1] / nf
        acc = np.mean((oof[vi] > 0.5) == y.iloc[vi].values)
        print(f"  [{label}] f{f+1} acc={acc*100:.1f}% i={m.best_iteration_}")
    print(f"  [{label}] OOF acc={np.mean((oof>0.5)==y.values)*100:.1f}%\n")
    return oof, pred

def cv_xgb_reg(X, y, Xt, params, sw, label, nf=N_FOLDS, clip=True):
    kf = KFold(nf, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X)); pred = np.zeros(len(Xt))
    for f, (ti, vi) in enumerate(kf.split(X)):
        m = xgb.XGBRegressor(**params)
        m.fit(X.iloc[ti], y.iloc[ti], sample_weight=sw[ti],
              eval_set=[(X.iloc[vi], y.iloc[vi])], verbose=False)
        oof[vi] = m.predict(X.iloc[vi]); pred += m.predict(Xt) / nf
        print(f"  [{label}] f{f+1} MAE={np.abs(oof[vi]-y.iloc[vi]).mean():.4f}")
    if clip: oof = np.maximum(oof, 0); pred = np.maximum(pred, 0)
    print(f"  [{label}] OOF MAE={np.abs(oof-y).mean():.4f}\n")
    return oof, pred

def cv_cat_reg(X, y, Xt, label, nf=N_FOLDS, clip=True):
    """CatBoost regression — handles categoricals natively."""
    kf = KFold(nf, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X)); pred = np.zeros(len(Xt))
    # CatBoost works better with actual categorical columns
    # We'll use the numeric features + embeddings
    for f, (ti, vi) in enumerate(kf.split(X)):
        m = CatBoostRegressor(
            iterations=1500, learning_rate=0.03, depth=6,
            loss_function='MAE', eval_metric='MAE',
            random_seed=SEED, verbose=False, thread_count=-1,
            early_stopping_rounds=100)
        m.fit(X.iloc[ti], y.iloc[ti],
              eval_set=(X.iloc[vi], y.iloc[vi]),
              verbose=False)
        oof[vi] = m.predict(X.iloc[vi]); pred += m.predict(Xt) / nf
        print(f"  [{label}] f{f+1} MAE={np.abs(oof[vi]-y.iloc[vi]).mean():.4f} i={m.best_iteration_}")
    if clip: oof = np.maximum(oof, 0); pred = np.maximum(pred, 0)
    print(f"  [{label}] OOF MAE={np.abs(oof-y).mean():.4f}\n")
    return oof, pred

# ================================================================
# 11. TRAINING
# ================================================================
print("=" * 55); print("LGB Poisson — team_goals"); print("=" * 55)
oof_lgb_t, pred_lgb_t = cv_lgb_reg(X_tr, train['team_goals'], X_te, LGB_POIS, sw, 'LGB_t')

print("=" * 55); print("LGB Poisson — opp_goals"); print("=" * 55)
oof_lgb_o, pred_lgb_o = cv_lgb_reg(X_tr, train['opp_goals'],  X_te, LGB_POIS, sw, 'LGB_o')

print("=" * 55); print("XGB Poisson — team_goals"); print("=" * 55)
oof_xgb_t, pred_xgb_t = cv_xgb_reg(X_tr, train['team_goals'], X_te, XGB_POIS, sw, 'XGB_t')

print("=" * 55); print("XGB Poisson — opp_goals"); print("=" * 55)
oof_xgb_o, pred_xgb_o = cv_xgb_reg(X_tr, train['opp_goals'],  X_te, XGB_POIS, sw, 'XGB_o')

print("=" * 55); print("LGB MAE — goal_difference"); print("=" * 55)
oof_lgb_gd, pred_lgb_gd = cv_lgb_reg(X_tr, train['gdt'], X_te, LGB_GD, sw, 'LGB_gd', clip=False)

print("=" * 55); print("XGB MAE — goal_difference"); print("=" * 55)
oof_xgb_gd, pred_xgb_gd = cv_xgb_reg(X_tr, train['gdt'], X_te, XGB_GD, sw, 'XGB_gd', clip=False)

# CatBoost
if HAS_CATBOOST:
    print("=" * 55); print("CatBoost — team_goals"); print("=" * 55)
    oof_cat_t, pred_cat_t = cv_cat_reg(X_tr, train['team_goals'], X_te, 'CAT_t')

    print("=" * 55); print("CatBoost — opp_goals"); print("=" * 55)
    oof_cat_o, pred_cat_o = cv_cat_reg(X_tr, train['opp_goals'],  X_te, 'CAT_o')

# Zero-inflation
print("=" * 55); print("LGB Binary — P(team_goals=0)"); print("=" * 55)
y_z_t = (train['team_goals'] == 0).astype(int)
oof_z_t, pred_z_t = cv_lgb_bin(X_tr, y_z_t, X_te, LGB_BIN, sw, 'ZERO_t')

print("=" * 55); print("LGB Binary — P(opp_goals=0)"); print("=" * 55)
y_z_o = (train['opp_goals'] == 0).astype(int)
oof_z_o, pred_z_o = cv_lgb_bin(X_tr, y_z_o, X_te, LGB_BIN, sw, 'ZERO_o')

# ================================================================
# 12. ENSEMBLE WEIGHTS — grid search
# ================================================================
print("=" * 55); print("Finding optimal ensemble weights..."); print("=" * 55)

best_ens = float('inf'); best_wt = {}

w_lgb_range = [0.35, 0.45, 0.55, 0.65]
w_cat_range  = [0.0, 0.15, 0.25] if HAS_CATBOOST else [0.0]

for w_lgb in w_lgb_range:
    for w_cat in w_cat_range:
        w_xgb = 1.0 - w_lgb - w_cat
        if w_xgb <= 0: continue
        if HAS_CATBOOST:
            ens_t = w_lgb*oof_lgb_t + w_xgb*oof_xgb_t + w_cat*oof_cat_t
            ens_o = w_lgb*oof_lgb_o + w_xgb*oof_xgb_o + w_cat*oof_cat_o
        else:
            ens_t = w_lgb*oof_lgb_t + (1-w_lgb)*oof_xgb_t
            ens_o = w_lgb*oof_lgb_o + (1-w_lgb)*oof_xgb_o
        ens_gd = w_lgb*oof_lgb_gd + (1-w_lgb)*oof_xgb_gd   # no catboost for GD

        for w_gd in [0.3, 0.5, 0.7]:
            gd_c  = w_gd * ens_gd + (1-w_gd) * (ens_t - ens_o)
            mid   = (ens_t + ens_o) / 2
            lam_t = np.maximum(mid + gd_c / 2, 0.05)
            lam_o = np.maximum(mid - gd_c / 2, 0.05)
            sc    = awmae(train, lam_t, lam_o)
            if sc < best_ens:
                best_ens = sc; best_wt = {'w_lgb':w_lgb,'w_xgb':w_xgb,'w_cat':w_cat,'w_gd':w_gd}
                print(f"  w_lgb={w_lgb:.2f} w_xgb={w_xgb:.2f} w_cat={w_cat:.2f} w_gd={w_gd:.1f} → {sc:.4f}")

print(f"\nBest weights: {best_wt}  AW-MAE={best_ens:.4f}")

def get_lambdas(oof=True):
    wl = best_wt['w_lgb']; wx = best_wt['w_xgb']; wc = best_wt['w_cat']; wg = best_wt['w_gd']
    if oof:
        t_lgb,o_lgb,gd_lgb = oof_lgb_t,oof_lgb_o,oof_lgb_gd
        t_xgb,o_xgb,gd_xgb = oof_xgb_t,oof_xgb_o,oof_xgb_gd
        t_cat,o_cat         = (oof_cat_t,oof_cat_o) if HAS_CATBOOST else (oof_lgb_t,oof_lgb_o)
    else:
        t_lgb,o_lgb,gd_lgb = pred_lgb_t,pred_lgb_o,pred_lgb_gd
        t_xgb,o_xgb,gd_xgb = pred_xgb_t,pred_xgb_o,pred_xgb_gd
        t_cat,o_cat         = (pred_cat_t,pred_cat_o) if HAS_CATBOOST else (pred_lgb_t,pred_lgb_o)
    ens_t  = wl*t_lgb + wx*t_xgb + wc*t_cat
    ens_o  = wl*o_lgb + wx*o_xgb + wc*o_cat
    ens_gd = wl*gd_lgb + (1-wl)*gd_xgb
    gd_c   = wg*ens_gd + (1-wg)*(ens_t - ens_o)
    mid    = (ens_t + ens_o) / 2
    lam_t_ = np.maximum(mid + gd_c / 2, 0.05)
    lam_o_ = np.maximum(mid - gd_c / 2, 0.05)
    return lam_t_, lam_o_

lam_t_oof, lam_o_oof = get_lambdas(oof=True)
lam_t_te,  lam_o_te  = get_lambdas(oof=False)

# ================================================================
# 13. ISOTONIC CALIBRATION — FIX SYSTEMATIC BIAS
#
# Model Poisson cenderung over-predict untuk tim kuat
# Isotonic regression maps raw predictions → calibrated values
# ================================================================
print("=" * 55); print("Isotonic calibration of λ..."); print("=" * 55)

def calibrate_lambda(lam_oof, y_true, lam_te):
    """Fit isotonic regression OOF → actual, apply to test."""
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(lam_oof, y_true)
    lam_cal_oof = iso.predict(lam_oof)
    lam_cal_te  = iso.predict(lam_te)
    lam_cal_oof = np.maximum(lam_cal_oof, 0.05)
    lam_cal_te  = np.maximum(lam_cal_te,  0.05)
    return lam_cal_oof, lam_cal_te

lam_t_cal_oof, lam_t_cal_te = calibrate_lambda(
    lam_t_oof, train['team_goals'].values, lam_t_te)
lam_o_cal_oof, lam_o_cal_te = calibrate_lambda(
    lam_o_oof, train['opp_goals'].values,  lam_o_te)

print(f"  Before calibration OOF AW-MAE: {awmae(train, lam_t_oof,     lam_o_oof):.4f}")
print(f"  After  calibration OOF AW-MAE: {awmae(train, lam_t_cal_oof, lam_o_cal_oof):.4f}")
print(f"  λ_team: {lam_t_oof.mean():.3f} → {lam_t_cal_oof.mean():.3f} (actual: {train['team_goals'].mean():.3f})")
print(f"  λ_opp : {lam_o_oof.mean():.3f} → {lam_o_cal_oof.mean():.3f} (actual: {train['opp_goals'].mean():.3f})\n")

# Choose best (calibrated vs uncalibrated)
if awmae(train, lam_t_cal_oof, lam_o_cal_oof) < awmae(train, lam_t_oof, lam_o_oof):
    lam_t_final_oof, lam_o_final_oof = lam_t_cal_oof, lam_o_cal_oof
    lam_t_final_te,  lam_o_final_te  = lam_t_cal_te,  lam_o_cal_te
    print("  Using CALIBRATED lambdas ✓")
else:
    lam_t_final_oof, lam_o_final_oof = lam_t_oof, lam_o_oof
    lam_t_final_te,  lam_o_final_te  = lam_t_te,  lam_o_te
    print("  Using RAW lambdas (calibration did not help on OOF)")

# ================================================================
# 14. FIT DIXON-COLES ρ — KEY INNOVATION
#
# Fitting ρ dari OOF predictions + actual goals
# ρ < 0 → lebih banyak 0-0, lebih sedikit 1-1 dan 2-2
# ================================================================
print("=" * 55); print("Fitting Dixon-Coles ρ parameter..."); print("=" * 55)

true_t_arr = train['team_goals'].values.astype(int)
true_o_arr = train['opp_goals'].values.astype(int)

# Grid search ρ on OOF (maximize log-likelihood of actual goals)
best_rho = -0.10; best_ll = float('inf')
for rho_try in np.arange(-0.25, 0.05, 0.025):
    ll = dc_neg_loglik(rho_try, lam_t_final_oof, lam_o_final_oof,
                       true_t_arr, true_o_arr)
    if ll < best_ll:
        best_ll = ll; best_rho = rho_try

# Fine-tune with scipy
result = minimize_scalar(
    lambda r: dc_neg_loglik(r, lam_t_final_oof, lam_o_final_oof,
                            true_t_arr, true_o_arr),
    bounds=(best_rho - 0.05, best_rho + 0.05),
    method='bounded')
best_rho = float(result.x)

print(f"  Optimal ρ = {best_rho:.4f}")
print(f"  ρ < 0 → increases P(0-0) and decreases P(1-0), P(0-1)")

# Verify Dixon-Coles improves OOF
def test_dc_oof(rho, lam_t, lam_o, df, sample_size=5000):
    """Quick test of DC optimal scorelines on subsample."""
    idx  = np.random.choice(len(lam_t), size=min(sample_size, len(lam_t)), replace=False)
    lt   = lam_t[idx]; lo = lam_o[idx]
    p    = dc_joint_pmf(lt, lo, rho)   # (N, G, G)
    pf   = p.reshape(len(idx), G*G)
    exp  = pf @ BL_FLAT
    best = np.argmin(exp, axis=1)
    opt_t= (best // G).astype(int)
    opt_o= (best %  G).astype(int)
    sub  = df.iloc[idx].reset_index(drop=True)
    return awmae_int(sub, opt_t, opt_o)

print(f"\n  Validation on 5K OOF subsample:")
np.random.seed(42)
sc_pois= test_dc_oof(0.0,   lam_t_final_oof, lam_o_final_oof, train)
sc_dc  = test_dc_oof(best_rho, lam_t_final_oof, lam_o_final_oof, train)
print(f"  Poisson (ρ=0.0)           : {sc_pois:.4f}")
print(f"  Dixon-Coles (ρ={best_rho:.3f}) : {sc_dc:.4f}")
print(f"  Improvement               : {(sc_pois-sc_dc)/sc_pois*100:.1f}%\n")

# ================================================================
# 15. FINAL TEST PREDICTIONS — Joint Match Optimization with Dixon-Coles
# ================================================================
print("=" * 55); print("Computing Dixon-Coles joint PMF for test..."); print("=" * 55)

# Compute expected loss using Dixon-Coles PMF
print("  Computing expected loss matrix (batch)...")
N_te    = len(lam_t_final_te)
BATCH   = 2000   # process in batches to manage memory
pred_team_r = np.zeros(N_te, dtype=int)
pred_opp_r  = np.zeros(N_te, dtype=int)

# Store full exp_loss for joint optimization
exp_loss_te = np.zeros((N_te, G, G), dtype=np.float32)
for start in range(0, N_te, BATCH):
    end = min(start + BATCH, N_te)
    lt  = lam_t_final_te[start:end]
    lo  = lam_o_final_te[start:end]
    p   = dc_joint_pmf(lt, lo, best_rho).astype(np.float32)
    pf  = p.reshape(end-start, G*G)
    exp_loss_te[start:end] = (pf @ BL_FLAT.astype(np.float32)).reshape(end-start, G, G)
    if (start // BATCH) % 5 == 0:
        print(f"    Processed {end:,}/{N_te:,}...")

print("  Joint match-level optimization...")
test_r    = test.reset_index(drop=True)
match_rows= test_r.groupby('match_id').apply(lambda x: x.index.tolist()).to_dict()

for rows in match_rows.values():
    if len(rows) == 2:
        i, j = rows[0], rows[1]
        # Joint: minimize E[loss_i(A,B)] + E[loss_j(B,A)]
        combined = exp_loss_te[i] + exp_loss_te[j].T   # (G, G)
        opt      = np.argmin(combined.ravel())
        pred_team_r[i] = opt // G
        pred_opp_r[i]  = opt %  G
        pred_team_r[j] = opt %  G
        pred_opp_r[j]  = opt // G
    else:
        for k in rows:
            opt = np.argmin(exp_loss_te[k].ravel())
            pred_team_r[k] = opt // G
            pred_opp_r[k]  = opt %  G

# ================================================================
# 16. VALIDASI & SIMPAN
# ================================================================
# Verify consistency
n_incons = sum(
    1 for rows in match_rows.values()
    if len(rows) == 2 and (
        pred_team_r[rows[0]] != pred_opp_r[rows[1]] or
        pred_opp_r[rows[0]]  != pred_team_r[rows[1]])
)
assert n_incons == 0, f"{n_incons} inconsistent pairs!"

sub = sample.copy()
sub['team_goals'] = pred_team_r
sub['opp_goals']  = pred_opp_r
assert len(sub) == len(sample)
assert (sub['team_goals'] >= 0).all() and (sub['opp_goals'] >= 0).all()
assert set(sub['Id']) == set(sample['Id'])

print(f"\n{'='*55}")
print("SUBMISSION SUMMARY")
print(f"{'='*55}")
n_win  = (sub['team_goals'] > sub['opp_goals']).sum()
n_draw = (sub['team_goals'] == sub['opp_goals']).sum()
n_loss = (sub['team_goals'] < sub['opp_goals']).sum()
print(f"Rows      : {len(sub):,}")
print(f"W/D/L     : {n_win:,} / {n_draw:,} / {n_loss:,}")
print(f"W/D/L %   : {n_win/len(sub)*100:.1f}% / {n_draw/len(sub)*100:.1f}% / {n_loss/len(sub)*100:.1f}%")
print(f"Train ref : 39.2% / 21.5% / 39.2%")
print(f"Avg goals : {sub['team_goals'].mean():.3f} / {sub['opp_goals'].mean():.3f}")
print(f"Train avg : 1.563 / 1.563")
print(f"0-gol %   : {(sub['team_goals']==0).mean()*100:.1f}%  (train ref: 30.2%)")
print(f"Consistency: {n_incons} inconsistent pairs")
print(f"\nDixon-Coles ρ  : {best_rho:.4f}")
print(f"DC improvement : {(sc_pois-sc_dc)/sc_pois*100:.1f}%")
print(f"\nTop 10 scorelines:")
print(sub.groupby(['team_goals','opp_goals']).size()
      .sort_values(ascending=False).head(10).to_string())

sub[['Id', 'team_goals', 'opp_goals']].to_csv('submission.csv', index=False)
print("\nDone! submission.csv siap diupload ke Kaggle.")

CatBoost found ✓
Train : 78,772 rows | 1872-11-30 → 2011-08-04
Test  : 42,422 rows  | 2011-08-06 → 2026-03-31
Precomputing AW-MAE loss tensor...
  Shape: (11, 11, 11, 11) ✓
Building encodings...
Building recency encodings...

Features: 68 | NaN train: 0 | NaN test: 0
LGB Poisson — team_goals
  [LGB_t] f1 MAE=1.0270 i=2999
  [LGB_t] f2 MAE=1.0230 i=3000
  [LGB_t] f3 MAE=1.0426 i=2818
  [LGB_t] f4 MAE=1.0381 i=2253
  [LGB_t] f5 MAE=1.0343 i=2994
  [LGB_t] OOF MAE=1.0330

LGB Poisson — opp_goals
  [LGB_o] f1 MAE=1.0270 i=2396
  [LGB_o] f2 MAE=1.0291 i=2890
  [LGB_o] f3 MAE=1.0279 i=2999
  [LGB_o] f4 MAE=1.0363 i=2994
  [LGB_o] f5 MAE=1.0437 i=2924
  [LGB_o] OOF MAE=1.0328

XGB Poisson — team_goals
  [XGB_t] f1 MAE=1.0262
  [XGB_t] f2 MAE=1.0233
  [XGB_t] f3 MAE=1.0440
  [XGB_t] f4 MAE=1.0328
  [XGB_t] f5 MAE=1.0339
  [XGB_t] OOF MAE=1.0321

XGB Poisson — opp_goals
  [XGB_o] f1 MAE=1.0250
  [XGB_o] f2 MAE=1.0299
  [XGB_o] f3 MAE=1.0276
  [XGB_o] f4 MAE=1.0359
  [XGB_o] f5 MAE=1.0426
  [XGB